# ARTI308 – Lab 6: Linear Regression on Adult Dataset  
**Dataset:** `adult.data`  

This notebook applies the same model used in the lab (**Linear Regression**) on our dataset.  
Because Linear Regression requires a **numerical target**, this notebook predicts:

## Target
`hours-per-week`

## Goal
Predict how many hours a person works per week based on demographic and work-related features.


## 1) Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn import metrics

sns.set_style("whitegrid")
%matplotlib inline


## 2) Load the dataset into a DataFrame

In [ ]:
columns = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race","sex",
    "capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

df = pd.read_csv("adult.data", names=columns, sep=",", skipinitialspace=True)
df.head()


## 3) Explore the data

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.columns


## 4) Basic data cleaning

In [ ]:
# Replace ? with NaN
df.replace("?", np.nan, inplace=True)

# Check missing values
df.isnull().sum()


In [ ]:
# Fill missing categorical values using mode
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Check again
df.isnull().sum()


## 5) Feature engineering  
We create a few simple features that may help the model:
- `net-capital` = capital-gain - capital-loss  
- `is-married` = 1 if married, else 0  
- `has-gain` = 1 if capital-gain > 0, else 0


In [ ]:
df["net-capital"] = df["capital-gain"] - df["capital-loss"]

married_values = ["Married-civ-spouse", "Married-AF-spouse", "Married-spouse-absent"]
df["is-married"] = df["marital-status"].apply(lambda x: 1 if x in married_values else 0)

df["has-gain"] = (df["capital-gain"] > 0).astype(int)

df.head()


## 6) Prepare the data for modeling  
Target = `hours-per-week`


In [ ]:
X = df.drop(columns=["hours-per-week"])
y = df["hours-per-week"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


## 7) Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=101
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


## 8) Build and train the model (Linear Regression)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)


## 9) Predictions

In [ ]:
predictions = model.predict(X_test)
predictions[:10]


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, predictions, alpha=0.4)
plt.xlabel("Actual Hours per Week")
plt.ylabel("Predicted Hours per Week")
plt.title("Actual vs Predicted")
plt.show()


## 10) Residual histogram

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot((y_test - predictions), bins=50)
plt.title("Residual Histogram")
plt.xlabel("Residuals")
plt.show()


## 11) Model evaluation

In [ ]:
print("MAE:", metrics.mean_absolute_error(y_test, predictions))
print("MSE:", metrics.mean_squared_error(y_test, predictions))
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test, predictions)))


## 12) Interpretation  
- **MAE** shows the average absolute prediction error.  
- **MSE** gives more penalty to larger errors.  
- **RMSE** is the square root of MSE and is easier to interpret because it is in the same unit as the target (`hours-per-week`).  
Lower values mean better model performance.


## 13) Conclusion  
In this lab, we applied **Linear Regression** on the Adult dataset.  
Since the original dataset is mostly used for classification, we selected a **numerical target** (`hours-per-week`) so that linear regression can be applied correctly.  
We cleaned missing values, created new features, encoded categorical variables, trained the model, and evaluated its performance using MAE, MSE, and RMSE.
